# Kiểm Thử Thực Nghiệm Pass 2 v2.4: PhoWhisper-medium ASR & Phân Tách Người Nói (Batch)

**Đề tài:** Hệ thống phiên âm, tóm tắt và hỏi đáp nội dung cuộc họp  
**Tài liệu tham chiếu:** `docs/02-research-tech/asr-diarization-survey.md` (Tầng 1: Pass 2 Offline Batch)  
**Môi trường thực thi:** Kaggle Notebook (hoặc Google Colab)  
**Lựa chọn GPU khuyến nghị trên Kaggle:** **GPU T4 x2** (NVIDIA Tesla T4 15GB VRAM) hoặc **GPU P100**  

---

### Mục tiêu phiên bản v2.4:
1. **Ưu tiên giữ đủ âm tiết trước khi lọc:**
   - Không dùng `repetition_penalty` quá cao hoặc `no_repeat_ngram_size` quá thấp, vì có thể làm mất từ lặp hợp lệ và làm méo câu tiếng Việt.
   - VAD chỉ là bộ lọc vùng có tiếng nói; mọi ngưỡng phải được kiểm chứng trên từng bản ghi, không tuyên bố lọc nhạc 100%.
2. **Bắt nhịp đối thoại Turn-Taking:**
   - Tách lượt nói khi khoảng lặng giữa hai từ >= 350ms hoặc khi gặp dấu câu (`.`, `?`, `!`) kèm khoảng nghỉ >= 200ms.
   - Phân tách dứt khoát câu hỏi của Giám đốc ("em là người đúng không?") và câu trả lời của Ứng viên ("em là người chứ em không phải AI").
3. **Diarization có nguồn và độ tin cậy rõ ràng:**
   - **Engine 1 (Pyannote Diarization):** Nạp qua GPU với tham số `min_speakers=2` chống hiện tượng under-clustering thành 1 người duy nhất.
   - **Engine 2 (optional heuristic):** Chỉ chạy khi đặt `ALLOW_HEURISTIC_DIARIZATION=1`; mặc định thiếu Pyannote/token sẽ giữ `SPEAKER_UNKNOWN` để không gán sai người nói.
4. **Chống hallucination có kiểm chứng:** Không retry ở temperature cao; đọc metadata của từng segment và cắt riêng suffix lặp liên tiếp khi có bằng chứng rõ ràng.
5. **Kaggle-safe:** Cài package lõi và package diarization độc lập; thiếu pyannote/token chỉ chuyển sang `SPEAKER_UNKNOWN` có cảnh báo, không làm notebook dừng giữa chừng.

## Bước 1: Kiểm Tra Cấu Hình Phần Cứng & Cài Đặt Thư Viện (Kaggle-safe)

In [ ]:
# Cài package bằng chính Python kernel đang chạy để tránh pip trỏ nhầm interpreter.
# Không ép downgrade numpy/torch của Kaggle; pyannote là optional để notebook vẫn chạy với SPEAKER_UNKNOWN.
import sys
import subprocess

def install_packages(packages: list[str], required: bool) -> bool:
    """Install notebook dependencies without hiding a required failure.

    Args:
        packages: Package specifications passed to pip.
        required: Whether a failed installation should stop the notebook.

    Returns:
        True when pip exits successfully, otherwise False.

    Raises:
        RuntimeError: If a required package installation fails.
    """
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", *packages],
        text=True,
        capture_output=True,
    )
    if result.returncode == 0:
        print(f"[CÀI ĐẶT] Đã sẵn sàng: {', '.join(packages)}")
        return True
    message = result.stderr[-2000:] if result.stderr else "Không có stderr"
    if required:
        raise RuntimeError(f"Không cài được package bắt buộc: {packages}\n{message}")
    print(f"[CẢNH BÁO] Bỏ qua package optional {packages}:\n{message}")
    return False

install_packages(["faster-whisper", "soundfile", "tabulate"], required=True)
install_packages(["pyannote.audio"], required=False)

# 2. Kiểm tra trạng thái phần cứng GPU
import torch
has_cuda = torch.cuda.is_available()
if has_cuda:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[XÁC NHẬN] Đang sử dụng GPU: {gpu_name} (VRAM: {vram_gb:.1f} GB)")
else:
    print("[CẢNH BÁO] Chưa bật GPU! Khuyến nghị bật GPU T4 x2 trong mục Accelerator bên phải.")

# 3. Kiểm tra nạp thư viện cốt lõi
PYANNOTE_AVAILABLE = False
PYANNOTE_IMPORT_ERROR = None
try:
    from faster_whisper import WhisperModel
    print("[XÁC NHẬN] faster-whisper đã nạp thành công.")
except Exception as e:
    raise RuntimeError(f"faster-whisper đã cài nhưng không import được: {e}") from e

try:
    from pyannote.audio import Pipeline
    PYANNOTE_AVAILABLE = True
    print("[XÁC NHẬN] pyannote.audio đã nạp thành công.")
except Exception as e:
    PYANNOTE_IMPORT_ERROR = f"{type(e).__name__}: {str(e)[:500]}"
    print(f"[LƯU Ý] pyannote.audio chưa khả dụng ({PYANNOTE_IMPORT_ERROR}). Hệ thống sẽ giữ SPEAKER_UNKNOWN; chỉ bật heuristic bằng ALLOW_HEURISTIC_DIARIZATION=1.")

## Bước 2: Thiết Lập Hugging Face Token (Hỗ Trợ HF_TOKEN trong Kaggle Secrets & Fallback An Toàn)

In [ ]:
import getpass
import os

HF_TOKEN = None
HF_TOKEN_SOURCE = None
SECRET_LOOKUP_ERRORS = []

# 1. Ưu tiên Kaggle Secrets để tránh dùng nhầm biến môi trường chứa token cũ.
# Hỗ trợ cả HF_TOKEN, HF_TOKEN2, HUGGINGFACE_TOKEN.
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for secret_key in ["HF_TOKEN", "HF_TOKEN2", "HUGGINGFACE_TOKEN", "hf_token"]:
        try:
            HF_TOKEN = user_secrets.get_secret(secret_key)
            if HF_TOKEN:
                HF_TOKEN_SOURCE = f"Kaggle Secret: {secret_key}"
                print(f"Đã tải thành công token từ Kaggle Secrets ({secret_key})!")
                break
        except Exception as exc:
            SECRET_LOOKUP_ERRORS.append(f"{secret_key}: {type(exc).__name__}: {str(exc)[:200]}")
except Exception as exc:
    SECRET_LOOKUP_ERRORS.append(f"kaggle_secrets: {type(exc).__name__}: {str(exc)[:200]}")

# 2. Khi chạy local hoặc không có Kaggle Secret, dùng biến môi trường.
if not HF_TOKEN:
    HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HF_TOKEN2")
    if HF_TOKEN:
        HF_TOKEN_SOURCE = "environment"

# 3. Không gọi getpass tự động vì Kaggle background có thể bị treo chờ nhập.
# Chỉ bật khi người dùng chủ động đặt ALLOW_INTERACTIVE_HF_TOKEN=1.
if not HF_TOKEN and os.getenv("ALLOW_INTERACTIVE_HF_TOKEN") == "1":
    HF_TOKEN = getpass.getpass("Nhập Hugging Face Token (Enter để bỏ qua): ").strip()

if HF_TOKEN:
    print("[XÁC NHẬN] HF_TOKEN đã được đọc (không in token); nguồn: " + str(HF_TOKEN_SOURCE))
    try:
        from huggingface_hub import HfApi
        account_info = HfApi(token=HF_TOKEN).whoami()
        account_name = account_info.get("name") or account_info.get("fullname") or "đã xác thực"
        print(f"[XÁC NHẬN] Token hợp lệ với Hugging Face, tài khoản: {account_name}")
        HfApi(token=HF_TOKEN).model_info("pyannote/speaker-diarization-community-1")
        print("[XÁC NHẬN] Token có thể truy cập model Pyannote community-1.")
    except Exception as exc:
        print(f"[CẢNH BÁO] Token hoặc quyền truy cập Pyannote chưa hợp lệ: {type(exc).__name__}: {str(exc)[:300]}")
else:
    print("[THÔNG BÁO] Không có HF_TOKEN -> speaker sẽ là SPEAKER_UNKNOWN; đặt token để chạy Pyannote.")
    if SECRET_LOOKUP_ERRORS:
        print("[CHI TIẾT] Không đọc được Kaggle Secret: " + " | ".join(SECRET_LOOKUP_ERRORS))

## Bước 3: Tự Động Quét Âm Thanh & Tiền Xử Lý Lọc Nhạc Nền (FFmpeg Bandpass + Loudnorm)

Áp dụng bộ lọc dải tần < 80Hz loại bỏ ù rè micro và chuẩn hóa âm lượng EBU R128 để cân bằng độ lớn giữa các người nói.

In [ ]:
import glob
import subprocess
import soundfile as sf

def auto_detect_audio(explicit_path: str | None = None) -> str:
    """Resolve one audio input deterministically.

    Args:
        explicit_path: User-provided input path.

    Returns:
        Existing audio path.

    Raises:
        FileNotFoundError: If an explicit path is invalid.
        RuntimeError: If automatic discovery finds multiple files.
    """
    if explicit_path:
        if not os.path.isfile(explicit_path):
            raise FileNotFoundError(f"Không tìm thấy file âm thanh: {explicit_path}")
        return explicit_path
    extensions = ["*.m4a", "*.mp3", "*.wav", "*.aac", "*.flac", "*.mp4", "*.ogg"]
    search_paths = [".", "/kaggle/input/**", "/kaggle/working/**"]
    
    found_files = []
    for sp in search_paths:
        for ext in extensions:
            matches = glob.glob(os.path.join(sp, ext), recursive=True)
            matches = [m for m in matches if "normalized" not in m and "live_input" not in m and "checkpoint" not in m]
            found_files.extend(matches)
            
    found_files = sorted(set(found_files))
    if len(found_files) == 1:
        chosen = found_files[0]
        print(f"[TỰ ĐỘNG PHÁT HIỆN] Đã tìm thấy tệp âm thanh của bạn: {chosen}")
        return chosen
    if len(found_files) > 1:
        raise RuntimeError("Tìm thấy nhiều file âm thanh; hãy truyền INPUT_AUDIO_PATH rõ ràng: " + ", ".join(found_files))
        
    print("[TẢI MẪU] Không tìm thấy file cá nhân, đang tự động tải tệp âm thanh tiếng Việt mẫu...")
    sample_urls = [
        "https://github.com/VinAIResearch/PhoWhisper/raw/main/assets/sample.wav",
        "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/vietnamese.wav",
    ]
    for sample_url in sample_urls:
        try:
            subprocess.run(["wget", "-q", "-O", "sample_meeting.wav", sample_url], check=True)
            return "sample_meeting.wav"
        except subprocess.CalledProcessError:
            continue
    raise FileNotFoundError("Không tải được audio mẫu")

INPUT_AUDIO_PATH = auto_detect_audio(os.getenv("INPUT_AUDIO_PATH"))
NORMALIZED_AUDIO_PATH = "normalized.wav"

# Chuẩn hóa bằng FFmpeg về chuẩn WAV Mono 16 kHz PCM 16-bit.
# Bản ghi mẫu có RMS thấp; loudnorm được bật mặc định nhưng có thể tắt bằng AUDIO_FILTER=raw để A/B.
AUDIO_FILTER = os.getenv("AUDIO_FILTER", "loudnorm").strip().lower()
if AUDIO_FILTER not in {"loudnorm", "raw"}:
    raise ValueError("AUDIO_FILTER chỉ nhận loudnorm hoặc raw")
cmd = [
    "ffmpeg", "-y", "-i", INPUT_AUDIO_PATH,
    "-ac", "1",                 # Mono (1 kênh)
    "-ar", "16000",             # Tần số mẫu 16 kHz
    "-c:a", "pcm_s16le",        # 16-bit PCM
    *( ["-af", "loudnorm=I=-20:TP=-1.5:LRA=11"] if AUDIO_FILTER == "loudnorm" else [] ),
    NORMALIZED_AUDIO_PATH
]
subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)

# Đọc âm thanh thành mảng NumPy float32 để truyền trực tiếp vào faster-whisper
audio_np, sr = sf.read(NORMALIZED_AUDIO_PATH, dtype="float32")
audio_duration_sec = len(audio_np) / sr
print(f"Chuẩn hóa thành công: {NORMALIZED_AUDIO_PATH}")
print(f"Thời lượng âm thanh: {audio_duration_sec:.2f} giây ({audio_duration_sec/60:.2f} phút) | Tần số mẫu: {sr} Hz")

## Bước 4: Nhận Dạng Tiếng Nói (PhoWhisper ASR) Triệt Tiêu Hallucination & Vòng Lặp Âm Nhạc

Cấu hình baseline ưu tiên bảo toàn âm tiết: VAD `threshold=0.45`, padding 240ms, temperature 0.0, metadata thresholds, repetition guard và word timestamps.

In [ ]:
import re
import time
from faster_whisper import WhisperModel

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
print(f"Thiết bị thực thi ASR: {device.upper()} (Compute: {compute_type})")

# Giữ cố định model theo yêu cầu; không cho phép fallback sang model khác.
ASR_MODEL_NAME = "nunu-willump/PhoWhisper-medium-ct2"
print(f"Đang khởi tạo mô hình ASR: {ASR_MODEL_NAME}...")
asr_model = WhisperModel(ASR_MODEL_NAME, device=device, compute_type=compute_type)
print(f"[THÀNH CÔNG] Đã tải mô hình ASR: {ASR_MODEL_NAME}")

def normalized_token(text: str) -> str:
    """Normalize one ASR word for repetition detection only.

    Args:
        text: Raw word text returned by faster-whisper.

    Returns:
        Lowercase token with surrounding punctuation removed.

    Raises:
        TypeError: If text is not a string.
    """
    if not isinstance(text, str):
        raise TypeError("ASR word phải là chuỗi")
    return re.sub(r"^[\W_]+|[\W_]+$", "", text.lower(), flags=re.UNICODE)


def repeated_suffix_start(words: list[dict], min_block_size: int = 4, max_block_size: int = 8) -> int | None:
    """Find a consecutive repeated phrase at the end of one ASR segment.

    This is deliberately lexical and conservative: a repeated block must be at
    least four words long and its second copy must be close to the segment end.
    It does not replace near-sounding Vietnamese words or use a business lexicon.

    """
    tokens = [normalized_token(word["w"]) for word in words]
    token_count = len(tokens)
    filler_tokens = {"hừm", "ừm", "hmm", "uh", "ờ", "à"}
    if token_count >= 4 and tokens[-1] in filler_tokens:
        run_start = token_count - 1
        while run_start > 0 and tokens[run_start - 1] == tokens[-1]:
            run_start -= 1
        if token_count - run_start >= 4:
            return run_start
    upper_bound = min(max_block_size, token_count // 2)
    for block_size in range(upper_bound, min_block_size - 1, -1):
        for second_start in range(max(0, token_count - 16), token_count - block_size + 1):
            first_start = second_start - block_size
            if first_start < 0:
                continue
            first_block = tokens[first_start:second_start]
            second_block = tokens[second_start:second_start + block_size]
            if first_block != second_block or not all(first_block):
                continue
            trailing_words = token_count - (second_start + block_size)
            if trailing_words <= 4:
                return first_start if first_start > 0 else second_start
    return None


def trim_repeated_suffix(words: list[dict]) -> tuple[list[dict], bool]:
    """Remove only a strongly repeated suffix from one ASR segment.

    Args:
        words: Word records containing w, s, e, and p fields.

    Returns:
        A pair containing the retained words and whether trimming occurred.

    Raises:
        ValueError: If a word record does not contain the required keys.
    """
    for word in words:
        if not {"w", "s", "e", "p"}.issubset(word):
            raise ValueError(f"Word record không hợp lệ: {word}")
    start = repeated_suffix_start(words)
    if start is None:
        return words, False
    return words[:start], True


print("\nBắt đầu nhận dạng PhoWhisper-medium với cấu hình chống hallucination...")
start_time = time.time()

# Không dùng hotword mặc định: hotword có thể làm decoder thiên về từ vựng không có trong audio.
# Có thể bật có kiểm soát bằng biến môi trường, ví dụ ASR_HOTWORDS="giám đốc, ứng tuyển".
ASR_HOTWORDS = os.getenv("ASR_HOTWORDS", "").strip() or None

segments_gen, info = asr_model.transcribe(
    audio_np,
    language="vi",
    beam_size=5,
    temperature=0.0,
    repetition_penalty=1.0,
    no_repeat_ngram_size=0,
    compression_ratio_threshold=2.4,
    log_prob_threshold=-1.0,
    no_speech_threshold=0.60,
    vad_filter=True,
    vad_parameters={
        "threshold": 0.45,
        "min_speech_duration_ms": 200,
        "max_speech_duration_s": 15.0,
        "min_silence_duration_ms": 500,
        "speech_pad_ms": 240,
    },
    word_timestamps=True,
    condition_on_previous_text=False,
    hallucination_silence_threshold=1.0,
    chunk_length=20,
    max_new_tokens=192,
    hotwords=ASR_HOTWORDS,
)

all_words = []
segment_diagnostics = []
accepted_segment_signatures = []
for seg in segments_gen:
    segment_words = []
    for word in seg.words or []:
        clean_word = word.word.strip()
        clean_word = re.sub(r"([.!?。！？])\1{2,}", r"\1", clean_word)
        if not clean_word or not normalized_token(clean_word) or normalized_token(clean_word) in {"unk", "<unk>", "[unk]"}:
            continue
        segment_words.append({
            "w": clean_word,
            "s": int(round(word.start * 1000)),
            "e": int(round(word.end * 1000)),
            "p": round(float(getattr(word, "probability", 0.0) or 0.0), 2),
        })

    compression_ratio = getattr(seg, "compression_ratio", None)
    avg_logprob = getattr(seg, "avg_logprob", None)
    no_speech_prob = getattr(seg, "no_speech_prob", None)
    segment_duration_ms = int(round((seg.end - seg.start) * 1000))
    duration_mismatch = len(segment_words) >= 3 and segment_duration_ms < len(segment_words) * 45
    metadata_rejected = (
        compression_ratio is not None and compression_ratio > 2.4
        and (avg_logprob is None or avg_logprob < -1.0)
    ) or (
        no_speech_prob is not None and no_speech_prob >= 0.85
        and (avg_logprob is None or avg_logprob < -1.0)
    ) or duration_mismatch
    if metadata_rejected:
        segment_diagnostics.append({
            "start_ms": int(round(seg.start * 1000)),
            "end_ms": int(round(seg.end * 1000)),
            "action": "drop_metadata_hallucination",
            "compression_ratio": compression_ratio,
            "avg_logprob": avg_logprob,
            "no_speech_prob": no_speech_prob,
            "duration_mismatch": duration_mismatch,
        })
        continue

    retained_words, trimmed = trim_repeated_suffix(segment_words)
    if trimmed:
        segment_diagnostics.append({
            "start_ms": int(round(seg.start * 1000)),
            "end_ms": int(round(seg.end * 1000)),
            "action": "trim_repeated_suffix",
            "compression_ratio": compression_ratio,
            "avg_logprob": avg_logprob,
            "no_speech_prob": no_speech_prob,
        })
    retained_signature = tuple(normalized_token(word["w"]) for word in retained_words)
    segment_start_ms = int(round(seg.start * 1000))
    segment_end_ms = int(round(seg.end * 1000))
    if retained_signature and accepted_segment_signatures:
        previous_signature, previous_end_ms = accepted_segment_signatures[-1]
        if retained_signature == previous_signature and segment_start_ms - previous_end_ms <= 500:
            segment_diagnostics.append({
                "start_ms": segment_start_ms,
                "end_ms": segment_end_ms,
                "action": "drop_adjacent_duplicate_segment",
            })
            continue
    if retained_signature:
        accepted_segment_signatures.append((retained_signature, segment_end_ms))
    all_words.extend(retained_words)

asr_duration = time.time() - start_time
rtf_asr = asr_duration / max(audio_duration_sec, 0.001)
print(f"Hoàn thành ASR trong: {asr_duration:.2f} giây (RTF: {rtf_asr:.4f})")
print(f"Tổng số từ nhận dạng được: {len(all_words)}")
if segment_diagnostics:
    print("Các segment đã được chặn/cắt do dấu hiệu hallucination:")
    for diagnostic in segment_diagnostics:
        print(diagnostic)
if all_words:
    print("5 từ đầu tiên:", all_words[:5])

## Bước 5: Phân Tách Người Nói (Pyannote chính, heuristic chỉ để baseline)

- **Engine 1 (Primary):** Pyannote Diarization với tham số ép `min_speakers=2` trên GPU.
- **Engine 2 (Optional):** Standalone Acoustic Clustering viết bằng **thuần PyTorch 100% (Zero Dependency)** chỉ chạy khi đặt `ALLOW_HEURISTIC_DIARIZATION=1`; mặc định thiếu token sẽ trả `SPEAKER_UNKNOWN`, không bịa nhãn speaker.

In [ ]:
import time
import numpy as np
import torch

speaker_turns = []
diarization_load_errors = []
dia_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
diarization_engine_used = None

# --- ENGINE 1: Pyannote Diarization (Primary - Khi có HF_TOKEN và package) ---
if HF_TOKEN and PYANNOTE_AVAILABLE:
    try:
        print("[ENGINE 1] Khởi tạo Pyannote Diarization Pipeline...")
        from pyannote.audio import Pipeline
        
        pipeline = None
        for p_model in ["pyannote/speaker-diarization-community-1", "pyannote/speaker-diarization-3.1"]:
            try:
                pipeline = Pipeline.from_pretrained(p_model, token=HF_TOKEN)
                if pipeline is not None:
                    print(f"[ENGINE 1] Đã nạp thành công pipeline: {p_model}")
                    break
            except Exception as exc:
                error_message = f"{type(exc).__name__}: {str(exc)[:500]}"
                diarization_load_errors.append({"model": p_model, "error": error_message})
                print(f"[ENGINE 1] Không tải được {p_model}: {error_message}")
                
        if pipeline is not None:
            pipeline.to(dia_device)
            start_time = time.time()
            
            # Chuyển tensor waveform lên cùng thiết bị GPU với pipeline
            waveform_tensor = torch.from_numpy(audio_np).unsqueeze(0).to(torch.float32).to(dia_device)
            
            # Ràng buộc theo kịch bản hai người; min_speakers đơn độc không đảm bảo đúng 2 cụm.
            diar_output = pipeline(
                {"waveform": waveform_tensor, "sample_rate": 16000},
                min_speakers=2,
                max_speakers=2
            )
            
            diar_result = getattr(diar_output, "exclusive_speaker_diarization", None)
            if diar_result is None:
                diar_result = getattr(diar_output, "speaker_diarization", diar_output)
            speaker_label_map = {}
            for turn, _, speaker in diar_result.itertracks(yield_label=True):
                if speaker not in speaker_label_map:
                    speaker_label_map[speaker] = f"SPEAKER_{len(speaker_label_map):02d}"
                speaker_turns.append((
                    int(round(turn.start * 1000)),
                    int(round(turn.end * 1000)),
                    speaker_label_map[speaker]
                ))
            
            dia_duration = time.time() - start_time
            print(f"[ENGINE 1] Hoàn thành Pyannote Diarization trong {dia_duration:.2f}s | Phát hiện {len(speaker_turns)} lượt nói.")
            diarization_engine_used = "Pyannote (Engine 1)"
        elif diarization_load_errors:
            print("[CẢNH BÁO] Pyannote không tải được pipeline. Hãy kiểm tra token, quyền truy cập model gated và phiên bản package.")
    except Exception as e:
        print(f"[CẢNH BÁO] Không thể chạy Pyannote Diarization ({e}).")
        print("==> Giữ SPEAKER_UNKNOWN. Chỉ bật ENGINE 2 bằng ALLOW_HEURISTIC_DIARIZATION=1 nếu chấp nhận kết quả cần review.")

# --- ENGINE 2: Standalone Acoustic Diarizer (Heuristic baseline; cần review) ---
if not speaker_turns and all_words and os.getenv("ALLOW_HEURISTIC_DIARIZATION") == "1":
    print("[ENGINE 2] Kích hoạt Standalone Acoustic Diarizer (Thuần PyTorch - Turn-Taking Tuned)...\n")
    start_time = time.time()
    
    # 1. Tách các cụm từ thành các lượt nói sơ bộ (Speech Turns)
    # Ngưỡng bắt nhịp đối thoại: khoảng lặng >= 350ms HOẶC dấu câu kết (. ? !) kèm khoảng nghỉ >= 200ms
    speech_turns_indices = []
    current_turn = [0]
    for i in range(1, len(all_words)):
        gap = all_words[i]["s"] - all_words[i-1]["e"]
        prev_w = all_words[i-1]["w"]
        is_punct = any(prev_w.endswith(p) for p in [".", "?", "!"])
        
        if gap >= 350 or (is_punct and gap >= 200):
            speech_turns_indices.append(current_turn)
            current_turn = [i]
        else:
            current_turn.append(i)
    if current_turn:
        speech_turns_indices.append(current_turn)
    
    # 2. Hàm trích xuất vector đặc trưng âm học bằng thuần PyTorch (FFT, Centroid, Rolloff, Subband Energy, Pitch)
    def extract_turn_features_torch(slice_np, sr=16000):
        if len(slice_np) < 256:
            return torch.zeros(20)
        sig = torch.from_numpy(slice_np).float()
        
        fft_vals = torch.abs(torch.fft.rfft(sig))
        freqs = torch.fft.rfftfreq(len(sig), 1.0 / sr)
        sum_fft = torch.sum(fft_vals) + 1e-9
        
        # Spectral Centroid (phân biệt rõ giọng nam trầm của GĐ và giọng cao của UV)
        centroid = torch.sum(freqs * fft_vals) / sum_fft
        # Spectral Rolloff
        cum_energy = torch.cumsum(fft_vals, dim=0)
        rolloff_idx = torch.searchsorted(cum_energy, 0.85 * cum_energy[-1])
        rolloff = freqs[min(rolloff_idx.item(), len(freqs) - 1)]
        
        # 16 logarithmic subbands (100Hz - 7500Hz)
        subband_edges = torch.tensor(np.geomspace(100, 7500, num=17)).float()
        subband_energies = []
        for i in range(16):
            mask = (freqs >= subband_edges[i]) & (freqs < subband_edges[i+1])
            energy = torch.mean(fft_vals[mask]**2) if mask.any() else torch.tensor(1e-9)
            subband_energies.append(torch.log(energy + 1e-9))
        subband_energies = torch.stack(subband_energies)
        
        # Pitch Autocorrelation trong dải tần số giọng người (75Hz - 300Hz)
        min_lag = int(sr / 300)
        max_lag = int(sr / 75)
        n = len(sig)
        f = torch.fft.rfft(sig, n=2*n)
        auto_corr = torch.fft.irfft(f * torch.conj(f))[:n]
        corr_win = auto_corr[min_lag:max_lag]
        best_lag = min_lag + torch.argmax(corr_win).item() if len(corr_win) > 0 else (min_lag + max_lag) // 2
        f0 = sr / max(best_lag, 1)
        
        zcr = torch.mean(torch.abs(torch.diff(torch.sign(sig)))) / 2.0
        
        feats = torch.cat([
            torch.tensor([centroid / 4000.0, rolloff / 7500.0, f0 / 300.0, zcr]),
            subband_energies / 10.0
        ])
        return feats
    
    # 3. Trích xuất đặc trưng cho từng lượt nói
    raw_features = []
    turn_ranges = []
    for turn_idx_list in speech_turns_indices:
        t_start = all_words[turn_idx_list[0]]["s"]
        t_end = all_words[turn_idx_list[-1]]["e"]
        
        # Bỏ qua các turn rác quá ngắn (< 300ms) của tiếng thở/tiếng ồn
        if (t_end - t_start) < 300 and len(turn_idx_list) <= 1:
            w_text = all_words[turn_idx_list[0]]["w"].lower().rstrip(".,?!")
            if w_text in ["hù", "hừm", "ờ", "unk"]:
                continue
                
        turn_ranges.append((t_start, t_end))
        s_idx = int(t_start * sr / 1000)
        e_idx = int(t_end * sr / 1000)
        slice_audio = audio_np[s_idx:e_idx]
        raw_features.append(extract_turn_features_torch(slice_audio, sr))
    
    # 4. Phân cụm 2 người nói bằng K-means++ thuần PyTorch (Zero Dependency, No Sklearn)
    def cluster_two_speakers_torch(X):
        mean = X.mean(dim=0, keepdim=True)
        std = X.std(dim=0, keepdim=True) + 1e-6
        X_norm = (X - mean) / std
        
        # Khởi tạo K-means++: 2 tâm cụm có khoảng cách xa nhau nhất
        dist_to_center = torch.norm(X_norm - X_norm.mean(dim=0), dim=1)
        idx0 = torch.argmax(dist_to_center).item()
        dist_to_idx0 = torch.norm(X_norm - X_norm[idx0], dim=1)
        idx1 = torch.argmax(dist_to_idx0).item()
        
        c0 = X_norm[idx0].clone()
        c1 = X_norm[idx1].clone()
        
        labels = torch.zeros(len(X_norm), dtype=torch.long)
        for _ in range(25):
            d0 = torch.norm(X_norm - c0, dim=1)
            d1 = torch.norm(X_norm - c1, dim=1)
            new_labels = (d1 < d0).long()
            if torch.equal(new_labels, labels):
                break
            labels = new_labels
            if (labels == 0).sum() > 0:
                c0 = X_norm[labels == 0].mean(dim=0)
            if (labels == 1).sum() > 0:
                c1 = X_norm[labels == 1].mean(dim=0)
                
        return labels.tolist()
    
    if len(raw_features) >= 2:
        X_tensor = torch.stack(raw_features)
        cluster_labels = cluster_two_speakers_torch(X_tensor)
    else:
        cluster_labels = [0] * len(raw_features)
    
    # 5. Đóng gói thành danh sách lượt phát biểu (speaker_turns)
    for (t_start, t_end), cl_id in zip(turn_ranges, cluster_labels):
        speaker_turns.append((t_start, t_end, f"SPEAKER_{cl_id:02d}"))
        
    dia_duration = time.time() - start_time
    print(f"[ENGINE 2] Hoàn thành baseline heuristic trong {dia_duration:.2f}s | Số cụm suy đoán: {len(set(cluster_labels))}")
    diarization_engine_used = "Standalone Acoustic Diarizer (Engine 2 - Explicitly enabled; review required)"
elif not speaker_turns:
    if not HF_TOKEN:
        diarization_engine_used = "Unavailable (HF_TOKEN missing)"
        print("[CẢNH BÁO] Kaggle chưa cung cấp HF_TOKEN cho kernel này. Kiểm tra Add-ons > Secrets và chạy lại từ đầu.")
    elif not PYANNOTE_AVAILABLE:
        diarization_engine_used = "Unavailable (pyannote.audio import failed)"
        print(f"[CẢNH BÁO] HF_TOKEN đã có nhưng pyannote.audio chưa import được: {PYANNOTE_IMPORT_ERROR}")
    elif diarization_load_errors:
        diarization_engine_used = "Unavailable (Pyannote pipeline load failed)"
        print("[CẢNH BÁO] HF_TOKEN và package đã có nhưng Pyannote không tải/chạy được pipeline; xem diarization_load_errors.")
    else:
        diarization_engine_used = "Unavailable (no speaker turns)"
        print("[CẢNH BÁO] Pyannote chạy nhưng không tạo speaker turn; speaker sẽ là SPEAKER_UNKNOWN.")

print(f"\n[XÁC NHẬN] Động cơ Diarization sử dụng: {diarization_engine_used}")
print(f"Tổng số lượt phát biểu: {len(speaker_turns)}")
unique_speakers = sorted(list(set([t[2] for t in speaker_turns])))
print(f"Danh sách người nói được phân tách: {unique_speakers}")

## Bước 6: Căn Chỉnh Word-to-Speaker Alignment & Đóng Gói Phân Đoạn (Turn-taking 500ms)

Gán người nói cho từng từ dựa vào diện tích giao cắt thời gian lớn nhất. Đóng gói phân đoạn theo ranh giới đổi người nói (`is_diff_spk`) và khoảng lặng >= 500ms.

In [ ]:
import uuid

def align_words_with_speakers(words: list[dict], turns: list[tuple[int, int, str]], max_gap_ms: int = 600) -> list[dict]:
    """Assign each word to the speaker turn with the largest overlap.

    Args:
        words: Word timestamps with `s` and `e` in milliseconds.
        turns: Diarization turns as `(start_ms, end_ms, speaker)`.
        max_gap_ms: Maximum distance for nearest-turn fallback.

    Returns:
        A copy of each word with a `speaker` label.

    Raises:
        ValueError: If a word has an invalid time range.
    """
    aligned_words = []
    for word in words:
        w_s, w_e = word["s"], word["e"]
        if w_e < w_s:
            raise ValueError(f"Mốc thời gian word không hợp lệ: {word}")
        best_speaker = None
        max_overlap = 0

        for t_s, t_e, spk in turns:
            overlap = min(w_e, t_e) - max(w_s, t_s)
            if overlap > max_overlap:
                max_overlap = overlap
                best_speaker = spk

        # Nếu từ rơi vào khoảng ngắt nghỉ, gán cho lượt gần nhất trong bán kính 600ms
        if best_speaker is None and len(turns) > 0:
            mid = (w_s + w_e) // 2
            nearest_turn = min(turns, key=lambda t: abs(mid - ((t[0] + t[1]) // 2)))
            dist = max(0, max(nearest_turn[0] - w_e, w_s - nearest_turn[1]))
            best_speaker = nearest_turn[2] if dist <= max_gap_ms else turns[0][2]
        elif best_speaker is None:
            best_speaker = "SPEAKER_UNKNOWN"

        aligned_words.append({**word, "speaker": best_speaker})
    return aligned_words

def has_speaker_overlap(start_ms: int, end_ms: int, speaker: str, turns: list[tuple[int, int, str]]) -> bool:
    """Return whether another speaker overlaps a segment in time."""
    return any(
        other_speaker != speaker and max(start_ms, turn_start) < min(end_ms, turn_end)
        for turn_start, turn_end, other_speaker in turns
    )

# 1. Gán nhãn cho từng từ
aligned_word_list = align_words_with_speakers(all_words, speaker_turns)

# 2. Gom các từ liên tiếp theo ranh giới đối thoại tự nhiên (Turn-taking Boundary)
final_segments = []
if aligned_word_list:
    current_spk = aligned_word_list[0]["speaker"]
    current_words = [aligned_word_list[0]]

    for w in aligned_word_list[1:]:
        gap_ms = w["s"] - current_words[-1]["e"]
        is_diff_spk = w["speaker"] != current_spk
        is_silence = gap_ms >= 500  # Ngắt nhịp tự nhiên khi im lặng >= 500ms
        
        # Ngắt câu tự nhiên nếu từ trước kết thúc bằng dấu chấm/hỏi/than và có khoảng nghỉ >= 250ms
        last_word_text = current_words[-1]["w"]
        has_punct = any(last_word_text.endswith(p) for p in [".", "?", "!"])
        is_sentence_break = has_punct and (gap_ms >= 250)

        if is_diff_spk or is_silence or is_sentence_break:
            text_content = " ".join([cw["w"] for cw in current_words])
            final_segments.append({
                "segment_id": f"seg_{uuid.uuid4().hex[:8]}",
                "start_ms": current_words[0]["s"],
                "end_ms": current_words[-1]["e"],
                "speaker": current_spk,
                "text": text_content,
                "words": [{"w": cw["w"], "s": cw["s"], "e": cw["e"], "p": cw["p"]} for cw in current_words],
                "overlap": has_speaker_overlap(current_words[0]["s"], current_words[-1]["e"], current_spk, speaker_turns),
                "confidence": round(sum([cw["p"] for cw in current_words]) / len(current_words), 2),
                "source": "final"
            })
            current_spk = w["speaker"]
            current_words = [w]
        else:
            current_words.append(w)

    # Lưu phân đoạn cuối cùng
    if current_words:
        text_content = " ".join([cw["w"] for cw in current_words])
        final_segments.append({
            "segment_id": f"seg_{uuid.uuid4().hex[:8]}",
            "start_ms": current_words[0]["s"],
            "end_ms": current_words[-1]["e"],
            "speaker": current_spk,
            "text": text_content,
            "words": [{"w": cw["w"], "s": cw["s"], "e": cw["e"], "p": cw["p"]} for cw in current_words],
            "overlap": has_speaker_overlap(current_words[0]["s"], current_words[-1]["e"], current_spk, speaker_turns),
            "confidence": round(sum([cw["p"] for cw in current_words]) / len(current_words), 2),
            "source": "final"
        })

print(f"Tổng số phân đoạn (segments) hội thoại tạo thành: {len(final_segments)}")

## Bước 7: Hậu Xử Lý Toàn Diện (Triệt Tiêu unk/Hù/Boilerplate), Bảng Kết Quả & Xuất JSON

In [ ]:
import json
import re
from tabulate import tabulate

def clean_transcript_text(text: str) -> str:
    """Normalize whitespace without changing recognized words.

    Args:
        text: Text reconstructed from the word timestamps.

    Returns:
        Whitespace-normalized transcript text.

    Raises:
        TypeError: If text is not a string.
    """
    if not isinstance(text, str):
        raise TypeError("text phải là chuỗi")
    return " ".join(text.split())

# 2. Lọc và chuẩn hóa toàn bộ phân đoạn
cleaned_segments = []
for s in final_segments:
    s["text"] = clean_transcript_text(s["text"])
    # Loại bỏ phân đoạn rỗng hoặc chỉ toàn từ rác sau khi làm sạch
    clean_words_only = re.sub(r"[^\w\s]", "", s["text"]).strip()
    if len(clean_words_only) > 0:
        cleaned_segments.append(s)

final_segments = cleaned_segments

# 3. In bảng đối chiếu kết quả
table_data = []
for s in final_segments:
    time_range = f"{s['start_ms']/1000:.1f}s - {s['end_ms']/1000:.1f}s"
    table_data.append([
        s["segment_id"],
        time_range,
        s["speaker"],
        s["text"],
        s["confidence"]
    ])

print("\n--- BẢNG KẾT QUẢ PHIÊN ÂM VÀ PHÂN TÁCH NGƯỜI NÓI (PASS 2 BATCH v2.4) ---")
if table_data:
    print(tabulate(table_data, headers=["Segment ID", "Thời gian", "Người nói", "Văn bản", "Độ tin cậy"], tablefmt="grid"))
else:
    print("Không có phân đoạn nào được sinh ra (file rỗng hoặc không có giọng nói).")

# 4. Lưu tệp JSON đầu ra chuẩn 100% tài liệu asr-diarization-survey.md
OUTPUT_JSON_PATH = "transcript_segments_output.json"
with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(final_segments, f, ensure_ascii=False, indent=2)

DIAGNOSTICS_JSON_PATH = "transcript_segments_diagnostics.json"
diagnostics_payload = {
    "asr": segment_diagnostics,
    "diarization_engine": diarization_engine_used,
    "speaker_turn_count": len(speaker_turns),
    "speaker_labels": unique_speakers,
    "diarization_load_errors": globals().get("diarization_load_errors", []),
}
with open(DIAGNOSTICS_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(diagnostics_payload, f, ensure_ascii=False, indent=2)

print(f"\n[XUẤT FILE THÀNH CÔNG] Đã lưu tệp kết quả: {OUTPUT_JSON_PATH}")
print(f"[DIAGNOSTICS] Đã lưu log chống hallucination: {DIAGNOSTICS_JSON_PATH}")

# 5. In mẫu cấu trúc phân đoạn đầu tiên để kiểm chứng đặc tả
if final_segments:
    print("\nMẫu cấu trúc phân đoạn đầu tiên:")
    print(json.dumps(final_segments[0], ensure_ascii=False, indent=2))